# NetSentinel — Expert 6: Data Exfiltration (VAE Anomaly Detector)

**Strategy (the defensible one):**
- **Train the VAE's notion of "normal" on a MODERN benign capture** → CIC-IoT2023 (2023, answers the concept-drift critique).
- **Test / benchmark on the recognized labeled set** → CSE-CIC-IDS2018 Infiltration day (contains labeled exfiltration flows).

The VAE sees only benign traffic during training. At inference, high reconstruction error = deviation from normal = candidate exfiltration.

---
## Datasets to attach in Kaggle (right sidebar → Add Input)

| Role | Kaggle search / dataset | What to add |
|---|---|---|
| **Train (modern benign)** | `CIC IoT 2023` (e.g. `akashdogra/cic-iot-2023`, or `madhavmalhotra/unb-cic-iot-dataset`) | the benign / normal CSV part |
| **Test (labeled exfil)** | `CSE-CIC-IDS2018` (e.g. `solarmainframe/ids-intrusion-csv` or `dhoogla/cse-cic-ids2018`) | the **Infiltration** day CSV (`...Infilteration...csv`) |
| *(optional 2nd test)* | `CIC-IDS2017` (`cicdataset/cicids2017`) | the **Thursday Infiltration** CSV |
| *(optional DNS exfil)* | `CIC-Bell-DNS-EXF-2021` | heavy/light exfil CSV |

> Column names differ slightly between releases. The `find_csv()` helper below auto-locates files by keyword, and the harmonizer keeps only the numeric flow columns common to both datasets, so you don't hand-edit paths.

**Enable GPU:** Settings → Accelerator → GPU (T4). The VAE is small; ~30 min is plenty.

In [ ]:
import os, glob, re, time, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', DEVICE)

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_csv(*keywords, root='/kaggle/input'):
    """Return CSV paths whose name contains ALL keywords (case-insensitive)."""
    hits = []
    for p in glob.glob(os.path.join(root, '**', '*.csv'), recursive=True):
        name = os.path.basename(p).lower()
        if all(k.lower() in name for k in keywords):
            hits.append(p)
    return sorted(hits)

# Sanity check: list everything Kaggle mounted
for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)[:50]:
    print(p)

## 1. Column harmonization
CIC flow datasets share a semantic feature set but not exact spelling. Normalize names, then keep the numeric intersection.

In [ ]:
def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    return df

LABEL_CANDIDATES = ['label', 'attack', 'class']

def load_concat(paths, nrows_each=None):
    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p, low_memory=False, nrows=nrows_each)
            frames.append(norm_cols(df))
            print(f'loaded {os.path.basename(p)} -> {df.shape}')
        except Exception as e:
            print('skip', p, e)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def get_label_col(df):
    for c in df.columns:
        if any(k == c or k in c for k in LABEL_CANDIDATES):
            return c
    return None

def clean_numeric(df, feat_cols):
    X = df[feat_cols].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna()
    return X, df.loc[X.index]

## 2. Load MODERN benign (train) + IDS2018 Infiltration (test)

In [ ]:
# --- TRAIN: modern benign from CIC-IoT2023 ---
train_paths = find_csv('benign') or find_csv('iot') or find_csv('normal')
print('TRAIN files:', train_paths)
df_train_raw = load_concat(train_paths, nrows_each=400_000)

# --- TEST: CIC-IDS2018 Infiltration day (benign + infiltration/exfil) ---
test_paths = find_csv('infil')  # matches 'Infilteration' / 'Infiltration'
print('TEST files:', test_paths)
df_test_raw = load_concat(test_paths)

assert len(df_train_raw) and len(df_test_raw), 'Check that datasets are attached and keywords match filenames.'

In [ ]:
# Keep benign rows only for TRAIN
ltr = get_label_col(df_train_raw)
if ltr:
    mask = df_train_raw[ltr].astype(str).str.lower().str.contains('benign|normal')
    df_train_raw = df_train_raw[mask]
print('train benign rows:', len(df_train_raw))

# TEST: build binary label  1 = exfil/infiltration attack, 0 = benign
lte = get_label_col(df_test_raw)
y_test = (~df_test_raw[lte].astype(str).str.lower().str.contains('benign|normal')).astype(int).values
print('test rows:', len(df_test_raw), '| attack rows:', int(y_test.sum()))

## 3. Feature engineering — the exfil ratio
Data exfiltration = far more bytes OUT than IN. The raw byte count alone makes the VAE flag any big upload; the **ratio** is the real signal.

In [ ]:
def add_exfil_features(df):
    df = df.copy()
    def pick(*opts):
        for o in opts:
            if o in df.columns: return o
        return None
    fwd_b = pick('totlen_fwd_pkts','total_length_of_fwd_packets','fwd_pkts_tot','total_fwd_packets')
    bwd_b = pick('totlen_bwd_pkts','total_length_of_bwd_packets','bwd_pkts_tot','total_backward_packets')
    fwd_p = pick('tot_fwd_pkts','total_fwd_packets','fwd_pkts_tot')
    bwd_p = pick('tot_bwd_pkts','total_backward_packets','bwd_pkts_tot')
    if fwd_b and bwd_b:
        df['exfil_byte_ratio'] = pd.to_numeric(df[fwd_b],errors='coerce') / (pd.to_numeric(df[bwd_b],errors='coerce')+1)
    if fwd_p and bwd_p:
        df['exfil_pkt_ratio'] = pd.to_numeric(df[fwd_p],errors='coerce') / (pd.to_numeric(df[bwd_p],errors='coerce')+1)
    return df

df_train_raw = add_exfil_features(df_train_raw)
df_test_raw  = add_exfil_features(df_test_raw)

# Common numeric feature set across both datasets (drop labels/ids)
drop_like = ('label','attack','class','flow_id','src_ip','dst_ip','source_ip','destination_ip','timestamp','src_port','dst_port')
def numeric_feats(df):
    return {c for c in df.columns if c not in drop_like and pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors='coerce'))}

feat_cols = sorted(numeric_feats(df_train_raw) & numeric_feats(df_test_raw))
print(f'{len(feat_cols)} shared features. exfil feats present:',
      [f for f in feat_cols if 'exfil' in f])

## 4. Clean + scale (fit scaler on benign only)

In [ ]:
Xtr, _ = clean_numeric(df_train_raw, feat_cols)
Xte, df_test_c = clean_numeric(df_test_raw, feat_cols)
yte = y_test[df_test_c.index.values] if len(y_test)==len(df_test_raw) else None

# hold out a benign validation split from TRAIN for thresholding
val_frac = 0.15
idx = np.random.permutation(len(Xtr)); cut = int(len(Xtr)*(1-val_frac))
Xtr_fit, Xtr_val = Xtr.iloc[idx[:cut]], Xtr.iloc[idx[cut:]]

scaler = StandardScaler().fit(Xtr_fit.values)
Xtr_fit_s = scaler.transform(Xtr_fit.values)
Xtr_val_s = scaler.transform(Xtr_val.values)
Xte_s     = scaler.transform(Xte.values)
print('train/val/test:', Xtr_fit_s.shape, Xtr_val_s.shape, Xte_s.shape)

## 5. The VAE (PyTorch)

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_in, d_hid=64, d_lat=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in,d_hid*2), nn.ReLU(),
                                 nn.Linear(d_hid*2,d_hid), nn.ReLU())
        self.mu, self.lv = nn.Linear(d_hid,d_lat), nn.Linear(d_hid,d_lat)
        self.dec = nn.Sequential(nn.Linear(d_lat,d_hid), nn.ReLU(),
                                 nn.Linear(d_hid,d_hid*2), nn.ReLU(),
                                 nn.Linear(d_hid*2,d_in))
    def forward(self,x):
        h = self.enc(x); mu,lv = self.mu(h), self.lv(h)
        z = mu + torch.randn_like(mu)*torch.exp(0.5*lv)
        return self.dec(z), mu, lv

def vae_loss(xr,x,mu,lv,beta=1.0):
    rec = nn.functional.mse_loss(xr,x,reduction='none').sum(1)
    kld = -0.5*torch.sum(1+lv-mu.pow(2)-lv.exp(),dim=1)
    return (rec + beta*kld).mean(), rec

model = VAE(Xtr_fit_s.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = DataLoader(TensorDataset(torch.tensor(Xtr_fit_s,dtype=torch.float32)),
                    batch_size=512, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(Xtr_val_s,dtype=torch.float32)),
                    batch_size=512, shuffle=False)

## 6. Train on benign ONLY

In [ ]:
EPOCHS = 30
history = {'train_loss': [], 'val_loss': []}
start_time = time.time()

for ep in range(EPOCHS):
    model.train(); t_loss = 0
    for (xb,) in train_loader:
        xb = xb.to(DEVICE)
        xr,mu,lv = model(xb)
        loss,_ = vae_loss(xr,xb,mu,lv)
        opt.zero_grad(); loss.backward(); opt.step()
        t_loss += loss.item()*len(xb)
        
    model.eval(); v_loss = 0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(DEVICE)
            xr,mu,lv = model(xb)
            loss,_ = vae_loss(xr,xb,mu,lv)
            v_loss += loss.item()*len(xb)
            
    tl = t_loss / len(train_loader.dataset)
    vl = v_loss / len(val_loader.dataset)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    
    if (ep + 1) % 5 == 0:
        print(f'Epoch {ep+1:02d} | Train Loss: {tl:.4f} | Val Loss: {vl:.4f}')

total_time = time.time() - start_time
print(f'\nDone in {total_time/60:.1f} min')

## 7. Threshold on benign validation, evaluate recall on exfil

In [ ]:
@torch.no_grad()
def recon_error(X):
    model.eval(); errs=[]
    for i in range(0,len(X),4096):
        xb = torch.tensor(X[i:i+4096],dtype=torch.float32,device=DEVICE)
        xr,mu,lv = model(xb)
        errs.append(((xr-xb)**2).sum(1).cpu().numpy())
    return np.concatenate(errs)

val_err  = recon_error(Xtr_val_s)
test_err = recon_error(Xte_s)

# threshold = 99th percentile of benign validation error (1% expected FP)
thr = np.percentile(val_err, 99)
pred = (test_err > thr).astype(int)
print(f'threshold(99th benign) = {thr:.2f}')

f1 = 0.0
auc = 0.0
if yte is not None:
    auc = roc_auc_score(yte, test_err)
    f1 = f1_score(yte, pred)
    print('ROC-AUC:', round(auc,4))
    print('F1-Score:', round(f1,4))
    print(classification_report(yte, pred, target_names=['benign','exfil']))

## 8. Training Curves & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('VAE Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Confusion Matrix
if yte is not None:
    cm = confusion_matrix(yte, pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Reds',
                xticklabels=['Benign', 'Exfiltration'], 
                yticklabels=['Benign', 'Exfiltration'], ax=axes[1])
    axes[1].set_title(f'Confusion Matrix (F1: {f1:.4f})', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No Test Labels Provided', ha='center', va='center')

plt.suptitle('Expert 6: Data Exfiltration (VAE)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'exfil_vae_graphs.png'), dpi=150)
plt.show()

## 9. Export for the NetSentinel backend (ONNX)

In [ ]:
import joblib, json
dummy = torch.randn(1, Xtr_fit_s.shape[1], device=DEVICE)
onnx_path = os.path.join(OUTPUT_DIR, 'expert6_vae.onnx')
scaler_path = os.path.join(OUTPUT_DIR, 'expert6_scaler.joblib')
meta_path = os.path.join(OUTPUT_DIR, 'expert6_meta.json')

torch.onnx.export(model, dummy, onnx_path,
                  input_names=['flow_features'], output_names=['recon','mu','logvar'],
                  dynamic_axes={'flow_features':{0:'batch'}}, opset_version=17)
joblib.dump(scaler, scaler_path)

metrics = {
    'model_name': 'NetSentinel Expert 6: Data Exfiltration',
    'model_type': 'Variational Autoencoder (VAE)',
    'version': '1.0.0',
    'f1': float(f1),
    'auc_roc': float(auc),
    'training_minutes': float(total_time / 60),
    'threshold': float(thr),
    'feature_cols': feat_cols,
    'data_sources': {
        'train_benign': 'CIC-IoT2023',
        'test_exfil': 'CSE-CIC-IDS2018 Infiltration'
    }
}
with open(meta_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print('\n' + '='*50)
print('MODEL CARD: Data Exfiltration VAE')
print('='*50)
print(f'  Arch:  Variational Autoencoder')
print(f'  Data:  CIC-IoT2023 + IDS2018')
print(f'  F1:    {f1:.4f}')
print(f'  AUC:   {auc:.4f}')
print(f'  ONNX:  {onnx_path}')
print('='*50)

In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_exfil_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, f)
        zf.write(fp, f)
        print(f'  {f} ({os.path.getsize(fp)/1024:.1f} KB)')

print(f'\nZip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
FileLink(zip_path)